# Text as data: sentiment analysis with TensorFlow

**Scientific question.** Can the language in a movie review predict whether its author rated the film positively or negatively?

This notebook builds an auditable text-classification pipeline: raw review $\rightarrow$ normalized tokens $\rightarrow$ integer sequence $\rightarrow$ learned embedding $\rightarrow$ pooled representation $\rightarrow$ binary logit. The goal is not merely to call `fit`; it is to preserve the meaning and provenance of every transformation.

> **Data contract.** The normal student route downloads and analyzes the genuine Stanford Large Movie Review Dataset (IMDB): 25,000 labeled training reviews and 25,000 labeled test reviews. Continuous integration explicitly activates a tiny, locally authored fixture to verify only the software path. CI evidence is never reported as evidence about movie sentiment.

## How to use this notebook

1. Connect to the internet and run the notebook. Keras downloads Stanford IMDB from its publisher and caches the archive; later runs reuse the cache.
2. Trace every tensor shape and inspect actual review lengths, labels, tokens, and errors before interpreting metrics.
3. Restart the kernel and run all cells. A trustworthy notebook must not depend on hidden state.
4. Interpret held-out errors; do not report training accuracy as model performance.

From the repository root, VS Code should use the **Rice DSM** kernel. If you already have an extracted corpus, set `RICE_DSM_IMDB_DIR` to its `aclImdb` directory. The CI-only smoke-test switch is set by the test harness and is not the student analysis route.

## Learning objectives

By the end, you should be able to:

- explain why text must be represented numerically without pretending that token IDs are quantitative measurements;
- split before learning a vocabulary, thereby preventing data leakage;
- distinguish normalization, tokenization, integerization, padding, masking, and out-of-vocabulary handling;
- explain what an embedding matrix learns and why pooling creates a fixed-width representation;
- train a binary classifier from logits using binary cross-entropy;
- choose and audit a decision threshold with a confusion matrix, precision, and recall; and
- identify sampling, labeling, negation, demographic, and deployment limitations.

## Kernel preflight: verify the process before the experiment

A notebook kernel is a long-running Python process. VS Code can remember a kernel from another project even after you open this repository. The next cell uses only Python's standard library, reports the interpreter actually running, and stops with exact recovery commands before importing TensorFlow.

In [ ]:
import os
import sys
from pathlib import Path

root_hint = os.environ.get("RICE_DSM_PROJECT_ROOT")
search_roots = ([Path(root_hint)] if root_hint else []) + [Path.cwd(), *Path.cwd().parents]
course_root = next((path.resolve() for path in search_roots if (path / "pyproject.toml").is_file()), None)
if course_root is None:
    raise RuntimeError("Open the course repository folder in VS Code, then restart this notebook.")
expected_prefix = (course_root / ".venv").resolve()
actual_prefix = Path(sys.prefix).resolve()
print("Python executable:", sys.executable)
print("Active environment:", actual_prefix)
print("Expected environment:", expected_prefix)
if actual_prefix != expected_prefix:
    raise RuntimeError(
        "Wrong notebook kernel. In the repository terminal run:\n"
        "  uv sync --locked\n"
        "  uv run python scripts/setup_course.py\n"
        "Then select 'Rice DSM' in VS Code and restart the kernel."
    )
assert Path(sys.executable).is_file()

## 1. Why this matters: language is structured, not tabular

A review is a variable-length sequence whose order, context, punctuation, and vocabulary can matter. A computer cannot optimize directly over strings, so we need a **representation pipeline**. Historically, sentiment systems progressed from hand-built lexicons and count vectors to learned dense word embeddings, recurrent and convolutional sequence models, and attention-based transformers. The progression changes how context is represented, but it does not remove the need for a valid data split, explicit label semantics, suitable metrics, and error analysis.

Here the label is $y=1$ for a positive review and $y=0$ for a negative review. The task is supervised binary classification, not a measurement of universal emotional truth. IMDB labels reflect the dataset's rating rule and population of reviewers.

In [ ]:
from __future__ import annotations

import random

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from tensorflow import keras

# Reproducibility spans Python, NumPy, TensorFlow, and deterministic kernels.
# A seed stabilizes one run; professional evidence still examines several seeds.
SEED = 438
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

print("TensorFlow:", tf.__version__)
print("Physical devices:", tf.config.list_physical_devices())
assert tf.__version__
assert SEED == 438

## 2. Acquire data with provenance and a cache

The normal student route obtains the genuine Stanford IMDB corpus directly from its publisher. `keras.utils.get_file` first checks Keras's local cache, downloads only when the archive is absent, and extracts it for reuse. Set `RICE_DSM_IMDB_DIR` only when you already have an extracted `aclImdb` directory.

We preserve the publisher's `train/` versus `test/` partition, then reserve validation observations *from the training partition*. We also take a deterministic, class-balanced subset so a live lecture has bounded runtime. A smaller sample changes uncertainty and attainable performance, so the sample limit is part of the experiment record.

Only automated CI sets `RICE_DSM_RUNNING_NOTEBOOK_TESTS=1`. In that mode, locally authored sentences exercise imports, shapes, and API contracts without a network request. Fixture metrics are software evidence, never evidence about real movie reviews.

In [ ]:
# Provenance belongs beside the loader, not in an untracked browser tab.
IMDB_ORIGIN = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
LIMIT_PER_CLASS = 1_000
CI_SMOKE_TEST = os.environ.get("RICE_DSM_RUNNING_NOTEBOOK_TESTS") == "1"


def read_review_directory(root: Path, split: str, limit_per_class: int):
    """Read the same bounded number of negative and positive reviews."""
    texts: list[str] = []
    labels: list[int] = []
    # Sorted paths make sample membership stable across operating systems.
    for label_name, label in (("neg", 0), ("pos", 1)):
        paths = sorted((root / split / label_name).glob("*.txt"))[:limit_per_class]
        texts.extend(path.read_text(encoding="utf-8") for path in paths)
        labels.extend([label] * len(paths))
    return np.asarray(texts, dtype=object), np.asarray(labels, dtype=np.int32)


assert IMDB_ORIGIN.startswith("https://ai.stanford.edu/")
assert LIMIT_PER_CLASS > 0

### Cache lookup and controlled download

There are three possible paths, in order: an explicitly supplied directory, Keras's standard cache, or a publisher download. The function returns `None` only when the CI switch is active. It raises rather than silently switching a student to fixture data after a failed download. That failure behavior protects the meaning of the analysis.

In [ ]:
def find_or_download_imdb() -> Path | None:
    """Resolve genuine IMDB data; bypass the network only in CI."""
    explicit = os.environ.get("RICE_DSM_IMDB_DIR")
    candidates = [Path(explicit).expanduser()] if explicit else []
    candidates.append(Path.home() / ".keras" / "datasets" / "aclImdb")

    for candidate in candidates:
        if (candidate / "train" / "pos").is_dir():
            return candidate.resolve()

    if CI_SMOKE_TEST:
        return None

    # get_file is cache-aware: it reuses the archive instead of downloading each run.
    archive = keras.utils.get_file(
        "aclImdb_v1.tar.gz", origin=IMDB_ORIGIN, extract=True
    )
    extracted = Path(archive).parent / "aclImdb"
    if not extracted.is_dir():
        raise FileNotFoundError(
            f"Downloaded {archive}, but no extracted aclImdb directory was found."
        )
    return extracted.resolve()

In [ ]:
# These authored sentences exist only to exercise the pipeline in offline CI.
# Students never opt into them as a substitute for genuine IMDB evidence.
FIXTURE_NEGATIVE = [
    "The story wandered and the final act never earned its dramatic ending.",
    "Strong actors could not rescue a dull script and awkward pacing.",
    "The jokes repeated until a short film felt painfully long.",
    "Beautiful lighting surrounded characters whose choices made no sense.",
    "I wanted tension, but predictable scenes removed every surprise.",
    "The soundtrack was loud while the emotional moments felt empty.",
    "Its promising premise disappeared beneath confusing exposition.",
    "Neither the mystery nor the romance reached a satisfying conclusion.",
    "A rushed adaptation flattened the most interesting characters.",
    "The sequel copied earlier scenes without their energy or charm.",
    "Uneven editing made conversations difficult to follow.",
    "The performance was committed, yet the film remained tedious.",
]
FIXTURE_POSITIVE = [
    "Patient storytelling made the quiet final scene deeply moving.",
    "The cast brought warmth and humor to an intelligent script.",
    "Inventive camera work made a familiar adventure feel new.",
    "Every clue mattered and the ending rewarded careful attention.",
    "The music supported the drama without overwhelming the characters.",
    "A thoughtful adaptation preserved the novel's difficult questions.",
    "The comedy balanced absurd jokes with genuine affection.",
    "Precise editing gave the action clarity, rhythm, and suspense.",
    "The lead performance transformed a simple plot into something memorable.",
    "Rich production design strengthened an already compelling story.",
    "The documentary was rigorous, humane, and visually confident.",
    "Its small observations accumulated into a powerful conclusion.",
]

### Preserve test data; create validation data from training data

`pool_text` comes only from the publisher's training directory. We split that pool into optimization and validation subsets. The publisher's test directory remains untouched until final evaluation. `stratify=pool_y` preserves the class proportion, and `random_state=SEED` makes membership reproducible.

In [ ]:
imdb_root = find_or_download_imdb()
if imdb_root is not None:
    pool_text, pool_y = read_review_directory(
        imdb_root, "train", LIMIT_PER_CLASS
    )
    test_text, test_y = read_review_directory(
        imdb_root, "test", LIMIT_PER_CLASS
    )
    data_source = "Stanford IMDB (bounded genuine subset)"
else:
    # Reaching this branch outside CI would silently invalidate the lesson.
    assert CI_SMOKE_TEST
    all_text = np.asarray(FIXTURE_NEGATIVE + FIXTURE_POSITIVE, dtype=object)
    all_y = np.asarray([0] * len(FIXTURE_NEGATIVE) + [1] * len(FIXTURE_POSITIVE))
    pool_text, test_text, pool_y, test_y = train_test_split(
        all_text, all_y, test_size=0.25, stratify=all_y, random_state=SEED
    )
    data_source = "CI fixture (not scientific evidence)"

train_text, val_text, train_y, val_y = train_test_split(
    pool_text, pool_y, test_size=0.2, stratify=pool_y, random_state=SEED
)

provenance = {
    "source": data_source,
    "publisher_url": IMDB_ORIGIN,
    "local_root": str(imdb_root) if imdb_root is not None else "CI fixture",
    "limit_per_class": LIMIT_PER_CLASS if imdb_root is not None else None,
    "sizes": {"train": len(train_y), "validation": len(val_y), "test": len(test_y)},
}
print(provenance)
assert len(train_text) == len(train_y)
assert set(np.unique(train_y)) == {0, 1}
assert set(train_text).isdisjoint(set(test_text))

### Inspect before transforming

Review length affects truncation, batching cost, and what evidence reaches the model. Class balance determines whether accuracy can hide a weak minority-class model. Duplicate overlap can make a test score optimistic. Reading examples can reveal HTML markup, encoding problems, label leakage, and language or genre shifts.

This is an intentionally public research corpus, so we print one bounded excerpt during exploration. Do not carry that habit into production logs: reviews, clinical notes, messages, and other user text may contain private or identifying information.

In [ ]:
# Whitespace counts are a quick diagnostic, not the model's final tokenization.
train_lengths = np.asarray([len(text.split()) for text in train_text])
class_counts = np.bincount(train_y, minlength=2)
length_percentiles = np.percentile(train_lengths, [50, 90, 95, 99]).round(1)

print("Class counts [negative, positive]:", class_counts)
print("Length percentiles [50, 90, 95, 99]:", length_percentiles)
print("Unique training reviews:", len(set(train_text)), "of", len(train_text))
print("Example label:", int(train_y[0]))
print(str(train_text[0])[:350])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(
    train_lengths, bins=min(25, max(5, len(train_lengths))), color="#0b7285"
)
axes[0].set(
    xlabel="Whitespace-separated words",
    ylabel="Reviews",
    title="Training review lengths",
)
axes[1].bar(
    ["negative", "positive"], class_counts, color=["#c92a2a", "#2b8a3e"]
)
axes[1].set(ylabel="Training reviews", title="Training labels")
plt.tight_layout()

assert train_lengths.min() > 0
assert class_counts.sum() == len(train_y)

## 3. From strings to integer sequences

`TextVectorization` combines several distinct operations:

1. **Normalization** makes selected surface forms consistent. The default used here lowercases and strips punctuation. This is a modeling choice: punctuation, capitalization, emoji, and negation can carry sentiment.
2. **Tokenization** divides text into units. We use whitespace-delimited words; subword tokenizers handle rare words differently.
3. **Vocabulary fitting** counts training tokens and assigns the most frequent entries integer IDs. Fit this vocabulary on training text only. Fitting on test text leaks information about the deployment distribution.
4. **OOV handling** maps unseen or excluded tokens to ID 1 (`[UNK]`). ID 0 is reserved for padding. An OOV token is not missing data; it means the chosen vocabulary cannot distinguish that token.
5. **Padding/truncation** creates a rectangular tensor. Short reviews receive zeros; long reviews lose tokens after the selected length. A mask tells compatible layers to ignore padding.

Token IDs are names, not measurements: token 200 is not twice token 100.

In [ ]:
MAX_TOKENS = 10_000 if imdb_root is not None else 500
SEQUENCE_LENGTH = 250 if imdb_root is not None else 40

# The layer stores learned preprocessing state: its vocabulary and token-to-ID map.
vectorizer = keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    standardize="lower_and_strip_punctuation",
    split="whitespace",
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    name="training_vocabulary",
)

# LEAKAGE BOUNDARY: adapt sees training strings and nothing from validation/test.
training_text_ds = tf.data.Dataset.from_tensor_slices(train_text).batch(64)
vectorizer.adapt(training_text_ds)
vocabulary = vectorizer.get_vocabulary()

print("Requested maximum vocabulary size:", MAX_TOKENS)
print("Learned vocabulary size:", len(vocabulary))
print("Fixed sequence length:", SEQUENCE_LENGTH)
assert 2 <= len(vocabulary) <= MAX_TOKENS

### Read one encoded sequence

The output has shape `(number of reviews, SEQUENCE_LENGTH)`. Every nonzero entry is a vocabulary address. ID 0 pads the end of a short review; ID 1 represents any token outside the learned vocabulary. Multiple unfamiliar words therefore collapse to the same representation. The embedding layer will later turn each address into a vector.

In [ ]:
probe_text = tf.constant([
    str(train_text[0]),
    "astonishingly qzxv unseenword",
])
encoded = vectorizer(probe_text)

print("Reserved entries [padding, OOV]:", vocabulary[:2])
print("Frequent learned tokens:", vocabulary[2:12])
print("Encoded tensor shape:", encoded.shape)
print("First review, first 16 IDs:", encoded[0, :16].numpy())
print("Unfamiliar-word probe, first 8 IDs:", encoded[1, :8].numpy())

assert vocabulary[0] == ""
assert vocabulary[1] == "[UNK]"
assert encoded.shape == (2, SEQUENCE_LENGTH)
assert 1 in encoded[1].numpy()  # At least one token is out of vocabulary.
assert int(encoded[1, -1]) == 0  # The fixed-width row ends in padding.

### Guided practice: audit truncation and OOV behavior

Compute the fraction of training reviews longer than `SEQUENCE_LENGTH`. Then create a sentence containing an invented token and confirm that ID 1 appears. If truncation is frequent, increasing the length preserves more evidence but increases memory and compute. If OOV frequency is high, investigate domain shift before merely enlarging the vocabulary.

In [ ]:
truncation_rate = float(np.mean(train_lengths > SEQUENCE_LENGTH))
probe_ids = vectorizer(
    tf.constant(["A qzxv performance with luminous acting"]),
)[0].numpy()

# Estimate OOV frequency after fitting. Padding is excluded from the denominator.
training_ids = vectorizer(tf.constant(train_text.tolist())).numpy()
non_padding = training_ids != 0
oov_rate = float(np.sum(training_ids == 1) / np.sum(non_padding))

print(f"Reviews truncated at {SEQUENCE_LENGTH} tokens: {truncation_rate:.1%}")
print(f"Non-padding training tokens mapped to [UNK]: {oov_rate:.1%}")
print("Probe IDs:", probe_ids[:8])
assert 0.0 <= truncation_rate <= 1.0
assert 0.0 <= oov_rate <= 1.0
assert 1 in probe_ids

## 4. Embeddings turn names into learned coordinates

Let $V$ be the vocabulary size and $d$ the embedding dimension. An embedding layer stores a matrix

$$E\in\mathbb{R}^{V\times d}.$$

For token ID $i$, lookup returns row $E_{i,:}$. These coordinates begin randomly and are adjusted by backpropagation for the prediction objective. They are not preordained semantic axes. Tokens that help make similar decisions may acquire nearby vectors, but a small supervised corpus does not guarantee a universal map of language.

For a padded sequence of length $L$, the embedding output has shape `(batch, L, d)`. `GlobalAveragePooling1D` averages unmasked token vectors to obtain one $d$-dimensional review vector. This is fast and instructive, but it mostly discards word order. Thus *good, not moving* and *not good, moving* can look undesirably similar. Convolutional, recurrent, and attention models preserve more local or long-range structure.

In [ ]:
BATCH_SIZE = 64 if imdb_root is not None else 8


def make_dataset(text, labels, *, training: bool):
    """Create a batched stream while preserving evaluation order."""
    # Labels become (observations, 1) to match the model's one-logit output.
    float_labels = labels.astype("float32")[:, None]
    dataset = tf.data.Dataset.from_tensor_slices((text, float_labels))
    if training:
        # Shuffle only optimization data; validation/test stay aligned with labels.
        dataset = dataset.shuffle(
            len(labels), seed=SEED, reshuffle_each_iteration=True
        )
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(train_text, train_y, training=True)
val_ds = make_dataset(val_text, val_y, training=False)
test_ds = make_dataset(test_text, test_y, training=False)

first_text_batch, first_label_batch = next(iter(train_ds))
print("Text batch shape:", first_text_batch.shape)
print("Label batch shape:", first_label_batch.shape)
assert first_text_batch.shape[0] == first_label_batch.shape[0]
assert first_label_batch.shape[1] == 1

### Build the representation and classifier one boundary at a time

The model accepts raw strings so the saved artifact can carry its own preprocessing. Shapes evolve as follows:

- raw text: `(batch,)`;
- integer IDs: `(batch, L)`;
- embeddings: `(batch, L, d)`;
- masked mean: `(batch, d)`;
- final logit: `(batch, 1)`.

`mask_zero=True` marks every padding ID. Keras propagates that Boolean mask to the pooling layer, so added zeros do not dilute the review vector. The mask does **not** hide OOV ID 1; an unknown token is still observed evidence.

In [ ]:
EMBEDDING_DIM = 32

text_input = keras.Input(shape=(), dtype=tf.string, name="review")
token_ids = vectorizer(text_input)

embedding_layer = keras.layers.Embedding(
    input_dim=len(vocabulary),
    output_dim=EMBEDDING_DIM,
    mask_zero=True,
    name="token_embedding",
)
embeddings = embedding_layer(token_ids)
review_vector = keras.layers.GlobalAveragePooling1D(
    name="masked_mean",
)(embeddings)
hidden = keras.layers.Dense(
    32, activation="relu", name="hidden_features"
)(review_vector)
hidden = keras.layers.Dropout(0.25, seed=SEED)(hidden)
logit = keras.layers.Dense(1, name="sentiment_logit")(hidden)
model = keras.Model(text_input, logit, name="imdb_sentiment_classifier")

# A small probe makes the shape contract visible before training.
probe_token_ids = vectorizer(probe_text)
probe_embeddings = embedding_layer(probe_token_ids)
probe_mask = embedding_layer.compute_mask(probe_token_ids)
print("Token IDs:", probe_token_ids.shape)
print("Embeddings:", probe_embeddings.shape)
print("Mask:", probe_mask.shape, "(True means real token)")
model.summary()

assert probe_embeddings.shape == (2, SEQUENCE_LENGTH, EMBEDDING_DIM)
assert probe_mask.shape == (2, SEQUENCE_LENGTH)
assert model.output_shape == (None, 1)

## 5. Binary logits, cross-entropy, and thresholds

The final layer emits a **logit** $z\in\mathbb{R}$. The sigmoid converts it to a modeled probability,

$$p_\theta(y=1\mid x)=\sigma(z)=\frac{1}{1+e^{-z}}.$$

For one target $y\in\{0,1\}$, binary cross-entropy is

$$\ell(z,y)=-y\log\sigma(z)-(1-y)\log(1-\sigma(z)).$$

A numerically stable logits implementation evaluates this without first forming probabilities near exactly zero or one. Its derivative is especially revealing:

$$\frac{\partial \ell}{\partial z}=\sigma(z)-y.$$

This is the same prediction-minus-target error structure encountered in logistic regression. A neural network changes how $z=f_\theta(x)$ is constructed; backpropagation carries this output error through embeddings and hidden layers.

The default decision rule $\hat y=\mathbf{1}[z\ge 0]$ is equivalent to a probability threshold of 0.5. That threshold is a policy choice, not part of the probability model.

In [ ]:
loss_function = keras.losses.BinaryCrossentropy(from_logits=True)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=loss_function,
    # This threshold is in logit space: logit 0 corresponds to probability 0.5.
    metrics=[keras.metrics.BinaryAccuracy(name="accuracy", threshold=0.0)],
)

# Check output and loss contracts before spending time on optimization.
initial_logits = model(first_text_batch, training=False)
initial_loss = loss_function(first_label_batch, initial_logits)
print("Initial logits shape:", initial_logits.shape)
print("Initial finite loss:", float(initial_loss))
assert initial_logits.shape == (len(first_label_batch), 1)
assert bool(tf.math.is_finite(initial_loss))

### Fit with a validation-controlled stopping rule

Adam updates parameters using training batches only. At each epoch, Keras evaluates the frozen epoch model on validation data. Early stopping retains the parameters from the smallest observed validation loss and prevents the final epoch from being chosen merely because it was last. Validation data influences model selection, so it is not an unbiased final test.

In [ ]:
EPOCHS = 8 if imdb_root is not None else 12
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
)
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stopping],
    verbose=0,
)

print("Epochs completed:", len(history.history["loss"]))
print("Best validation loss:", min(history.history["val_loss"]))
assert len(history.history["loss"]) >= 1
assert np.isfinite(history.history["loss"]).all()

### Read learning curves as paired evidence

Training loss diagnoses optimization on data that updates the parameters. Validation loss estimates generalization during model selection. A widening gap—training improving while validation worsens—is evidence of overfitting. Accuracy is thresholded and less sensitive than cross-entropy to changes in confidence, so inspect both rather than declaring success from one curve.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(history.history["loss"], marker="o", label="train")
axes[0].plot(history.history["val_loss"], marker="o", label="validation")
axes[0].set(
    xlabel="Epoch",
    ylabel="Binary cross-entropy",
    title="Optimization and generalization",
)
axes[0].legend()

axes[1].plot(history.history["accuracy"], marker="o", label="train")
axes[1].plot(history.history["val_accuracy"], marker="o", label="validation")
axes[1].set(
    xlabel="Epoch",
    ylabel="Accuracy at probability 0.5",
    ylim=(0, 1),
    title="Thresholded decisions",
)
axes[1].legend()
plt.tight_layout()

## 6. Worked example: held-out evaluation is more than accuracy

Accuracy answers, "What fraction of thresholded decisions were correct?" It hides which class was harmed. Precision asks, "Among predicted positives, what fraction were positive?" Recall asks, "Among actual positives, what fraction did we detect?" Their relevance depends on consequences. Mistaking an unfavorable review for a favorable one may distort a recommendation; missing critical feedback may hide a product problem.

Architecture, checkpoint, and threshold choices belong to validation data. Below, the validation curve visualizes the threshold tradeoff. We retain the predeclared probability threshold 0.5 rather than inventing an operational cost for this classroom task. Only after that policy is frozen do we touch the test set once. In CI, every resulting number is a mechanics check rather than a population claim.

In [ ]:
# Validation data may guide a policy; test data may not.
val_logits = model.predict(val_ds, verbose=0).reshape(-1)
val_probabilities = tf.sigmoid(val_logits).numpy()
thresholds = np.linspace(0.2, 0.8, 25)
val_precisions = [
    precision_score(val_y, val_probabilities >= threshold, zero_division=0)
    for threshold in thresholds
]
val_recalls = [
    recall_score(val_y, val_probabilities >= threshold, zero_division=0)
    for threshold in thresholds
]

plt.figure(figsize=(7, 3.5))
plt.plot(thresholds, val_precisions, label="validation precision")
plt.plot(thresholds, val_recalls, label="validation recall")
plt.axvline(0.5, color="black", linestyle="--", linewidth=1, label="predeclared")
plt.xlabel("Positive-class probability threshold")
plt.ylabel("Metric")
plt.ylim(0, 1.02)
plt.title("Validation evidence: thresholds trade precision for recall")
plt.legend()
plt.show()

assert val_probabilities.shape == val_y.shape

In [ ]:
DECISION_THRESHOLD = 0.5  # Frozen before inspecting test outcomes.

test_logits = model.predict(test_ds, verbose=0).reshape(-1)
test_probabilities = tf.sigmoid(test_logits).numpy()
test_predictions = (test_probabilities >= DECISION_THRESHOLD).astype(int)

print(classification_report(
    test_y,
    test_predictions,
    target_names=["negative", "positive"],
    zero_division=0,
))
ConfusionMatrixDisplay(
    confusion_matrix(test_y, test_predictions),
    display_labels=["negative", "positive"],
).plot(cmap="Blues", colorbar=False)
plt.title(f"Final held-out decisions — {data_source}")
plt.show()

assert test_probabilities.shape == test_y.shape
assert np.all((0.0 <= test_probabilities) & (test_probabilities <= 1.0))

## 7. Debugging and error analysis

Inspect false positives and false negatives rather than treating the confusion matrix as a decorative endpoint. Look for negation ("not good"), sarcasm, mixed sentiment, plot words correlated with labels, truncation, names mapped to OOV, and reviews whose star rating conflicts with their prose.

**Common failure modes**

- Adapting the vocabulary on all data: test-distribution leakage.
- Passing probabilities to `BinaryCrossentropy(from_logits=True)`: a mismatched output/loss contract.
- Treating padding ID 0 as a word: length becomes a spurious signal.
- Evaluating dropout without inference mode: unstable predictions. Keras handles this distinction inside `fit`, `evaluate`, and `predict`.
- Reading accuracy alone: asymmetric errors remain invisible.
- Interpreting embedding proximity causally: learned association is neither causation nor a stable social meaning.

In [ ]:
error_indices = np.flatnonzero(test_predictions != test_y)
print(f"Misclassified reviews: {len(error_indices)} of {len(test_y)}")

if len(error_indices) == 0:
    message = (
        "No errors appeared in this split. Do not infer perfection from a tiny CI fixture; "
        "use genuine IMDB data for substantive analysis."
    )
    print(message)
else:
    for index in error_indices[:5]:
        review = str(test_text[index])
        ids = vectorizer(tf.constant([review]))[0].numpy()
        observed_ids = ids[ids != 0]
        oov_count = int(np.sum(observed_ids == 1))
        error_type = "false positive" if test_predictions[index] == 1 else "false negative"
        print(
            f"{error_type} | truth={test_y[index]} | "
            f"P(positive)={test_probabilities[index]:.3f} | "
            f"retained_tokens={len(observed_ids)} | OOV={oov_count}"
        )
        print(review[:400])
        print("Questions: negation? mixed sentiment? OOV? truncation? shortcut token?")
        print("-" * 88)

## 8. Guided investigation

Choose one controlled comparison and write down the hypothesis *before* fitting:

- mean pooling versus a 1D convolution or bidirectional recurrent layer;
- 5,000 versus 20,000 vocabulary entries;
- 100 versus 400 tokens per review; or
- the default normalization versus a custom normalizer that preserves `!` or marks negation.

Hold the split, seed, metric definitions, and test protocol fixed. Record parameter count, training time, validation loss, and class-specific errors. A larger score alone is not an explanation.

In [ ]:
# A zero-training shape check for a proposed sequence-aware replacement.
probe_input = keras.Input(shape=(SEQUENCE_LENGTH,), dtype=tf.int64)
probe_embedding = keras.layers.Embedding(len(vocabulary), 16, mask_zero=True)(probe_input)
probe_features = keras.layers.Bidirectional(keras.layers.GRU(8))(probe_embedding)
probe_model = keras.Model(probe_input, probe_features)
print("Proposed feature shape:", probe_model.output_shape)
assert probe_model.output_shape == (None, 16)

## 9. Independent extension exercise

Use the real IMDB dataset and answer one specific question, such as: *Does preserving more of each review improve recall on negative reviews enough to justify the added latency?* Your report must include:

1. a data card naming the source, label construction, split, license/terms, and exclusions;
2. an explicit preprocessing contract and proof that vocabulary fitting used training data only;
3. a justified baseline and one controlled model change;
4. loss curves, a confusion matrix, precision/recall, and at least ten categorized errors;
5. timing and model-size evidence; and
6. a conclusion whose strength matches the held-out evidence.

Do not tune repeatedly on the test set. If you use a different corpus, discuss consent, privacy, dialect, annotation quality, and foreseeable misuse.

## 10. Responsible limitations and professional practice

Sentiment is contextual. A model trained on English-language IMDB reviews may fail on other languages, short social posts, contemporary slang, disability-related language, reclaimed terms, or reviews from a different platform. The dataset excludes neutral ratings, so a binary model is forced to express certainty on ambiguous or mixed reviews. Correlations with genre, actor names, or era may produce shortcuts.

For deployment, version the raw-data manifest, vocabulary, normalization code, model weights, threshold, and metric definitions together. Monitor OOV rate, token-length distributions, score distributions, latency, missing/empty inputs, and delayed label quality. Log aggregate diagnostics—not raw user text—unless retention, consent, security, and privacy requirements explicitly permit it. Include a human escalation path when decisions affect people.

## Retrieval practice

1. Why is adapting the vocabulary before splitting a form of leakage even though labels are not used?
2. What do IDs 0 and 1 mean in this pipeline?
3. Why is an embedding lookup different from treating an integer token ID as a scalar feature?
4. Derive $\partial\ell/\partial z=\sigma(z)-y$ for binary cross-entropy from logits.
5. What information does mean pooling discard?
6. Why can the best threshold differ from 0.5?
7. Which result in this notebook is invalid as scientific evidence when the fixture is active?

## Takeaway

A text classifier is a chain of contracts. The corpus defines whose language and which labels are represented; the split defines what information training may use; the vectorizer defines what the model can distinguish; the architecture defines how sequence evidence is combined; the loss defines the fitted objective; and the threshold defines an operational decision. TensorFlow automates differentiation and optimized execution, but it cannot choose or validate these contracts for us.

## Further reading

- [Stanford Large Movie Review Dataset](https://ai.stanford.edu/~amaas/data/sentiment/)
- [Keras text classification guide](https://keras.io/examples/nlp/text_classification_from_scratch/)
- [TensorFlow `TextVectorization`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/TextVectorization)
- [TensorFlow masking and padding guide](https://www.tensorflow.org/guide/keras/understanding_masking_and_padding)
- [Python data model](https://docs.python.org/3/reference/datamodel.html)